In [ ]:
# Open-source sanitization: cached outputs cleared to remove credential and session traces.
import json
jsonl_file="/data/wangsl/checkpoints/reward_hacking_mre/train_deepswe-debug-rejection_sample-rollout-n8-ppio-16parallels/chat_completions/1.jsonl"

datas = []
with open(jsonl_file, "r") as f:
    for line in f:
        data = json.loads(line)
        datas.append(data)

In [ ]:
import asyncio
import time
from openai import OpenAI
from concurrent.futures import ThreadPoolExecutor
# Open-source sanitization: hardcoded credential or private token removed.
client = OpenAI(api_key='', base_url='https://api.ppinfra.com/openai/v1')
model = "qwen/qwen3-235b-a22b-fp8"

print(client.chat.completions.create(model=model, messages=[{"role": "user", "content": "你好"}]).choices[0].message.content)

In [ ]:
import asyncio
import time
from openai import OpenAI
from concurrent.futures import ThreadPoolExecutor
prompts = []
for data in datas:
    prompt = """以下是一个 AI 助手和用户的对话。用户让 AI 完成 SWE 任务，但是 AI 最终做错了。
    对话内容如下：
"""
    for chat in data:
        role = chat["role"]
        content = chat["content"]
        prompt += f"#" * 80 + f" {role} " + f"#" * 80 + "\n"
        prompt += content + "\n\n"
    prompt += "请你简要判断每一步中 AI 犯了什么错误？并总结整个流程失败的原因。"
    prompts.append(prompt)

def judge_with_retry(prompt):
    backoff = 10
    for _ in range(10):
        try:
            response = client.chat.completions.create(
                model=model,
                messages=[{"role": "user", "content": prompt}]
            )
            print(response.choices[0].message.content)
            return response.choices[0].message.content
        except Exception as e:
            print(f"Error: {e}")
            time.sleep(backoff)
            backoff = min(backoff*2, 40)
    return None

with ThreadPoolExecutor(max_workers=4) as executor:
    results = list(executor.map(judge_with_retry, prompts))


In [ ]:
import asyncio
import time
from openai import OpenAI
from concurrent.futures import ThreadPoolExecutor
prompts = []
for data in datas:
    prompt = """以下是一个 AI 助手和用户的对话。请你判断最终 AI 助手是否成功运行 reproduce_issue.py 脚本。
    对话内容如下：
"""
    for chat in data:
        role = chat["role"]
        content = chat["content"]
        prompt += f"#" * 80 + f" {role} " + f"#" * 80 + "\n"
        prompt += content + "\n\n"
    prompt += "请判断最终 AI 助手是否成功运行 reproduce_issue.py 脚本，如果成功运行，请返回 True，否则返回 False。并告诉我做出判断的依据是什么"
    prompts.append(prompt)

def judge_with_retry(prompt):
    backoff = 10
    for _ in range(10):
        try:
            response = client.chat.completions.create(
                model=model,
                messages=[{"role": "user", "content": prompt}]
            )
            print(response.choices[0].message.content)
            return response.choices[0].message.content
        except Exception as e:
            print(f"Error: {e}")
            time.sleep(backoff)
            backoff = min(backoff*2, 40)
    return None

with ThreadPoolExecutor(max_workers=4) as executor:
    results = list(executor.map(judge_with_retry, prompts))


In [ ]:
def check_result(response):
    if not isinstance(response, str):
        return False
    if 'true' in response.lower():
        return True
    else:
        return False

In [ ]:
from rllm.utils import colorful_print
for idx, result in enumerate(results):
    colorful_print(("#"*50+f" {idx} "+"#"*50+'\n')*3, "green" if check_result(result) else "red")
    print(result)

In [ ]:
with open('./wandb/latest-run/files/output.log', 'r', encoding='utf8') as f:
    log = f.read()

complete_reasons = ['ENV_DONE', 'TRUNCATION', 'MAX_STEP']
total_traj = 0
done = 0
trunc = 0
max_step = 0
for line in log.split('\n'):
    if any([r in line for r in complete_reasons]):
        total_traj += 1
        if 'ENV_DONE' in line:
            done += 1
        elif 'TRUNCATION' in line:
            trunc += 1
        elif 'MAX_STEP' in line:
            max_step += 1

print(done / total_traj * 100)
print(trunc / total_traj * 100 )
print(max_step / total_traj * 100)

In [ ]:
for idx, data in enumerate(datas):
    if any(['Finished' in ''.join(chat.values()) and 'Please submit your answer NOW' not in ''.join(chat.values()) for chat in data]):
        for chat in data:
            role = chat["role"]
            content = chat["content"]
            colorful_print(("#"*50 + f" {role} " + "#"*50), "blue")
            print(content)
        break